In [0]:
# Create widgets
dbutils.widgets.text("source_system", "")
dbutils.widgets.text("table_name", "")
dbutils.widgets.text("landing_timestamp", "")
dbutils.widgets.text("source_file_format", "")
# Get widget values into variables
source_system = dbutils.widgets.get("source_system")
table_name = dbutils.widgets.get("table_name")
landing_timestamp = dbutils.widgets.get("landing_timestamp")
source_file_format = dbutils.widgets.get("source_file_format")

In [0]:

dbutils.fs.ls('/Volumes/adb_caren/landing/landing-volume/neon_postgres/hospitals/2026-11-02_08:12:30/')

In [0]:
if source_file_format == 'csv':
    landing_df = spark.read.format('csv').option('header', 'true').option('inferSchema', 'true').load(f'/Volumes/adb_caren/landing/landing-volume/{source_system}/{table_name}/{landing_timestamp}/')
else:
    landing_df = spark.read.format('parquet').option('header', 'true').option('inferSchema', 'true').load(f'/Volumes/adb_caren/landing/landing-volume/{source_system}/{table_name}/{landing_timestamp}/')

In [0]:
from pyspark.sql import functions as F

updated_df = landing_df.withColumn("landing_timestamp", F.lit(landing_timestamp)).withColumn('insert_timestamp', F.current_timestamp())

In [0]:
updated_df = updated_df.withColumn("landing_timestamp", F.to_timestamp("landing_timestamp", "yyyy-MM-dd_HH:mm:ss"))
updated_df.write.mode("append").saveAsTable(f'adb_caren.bronze.{table_name}')